# 🧠 GNN Quantum Circuit Optimizer — Training Notebook

Train a Graph Attention Network (GAT) to predict optimization improvement ratios for quantum circuits.

**Dataset**: ~10,000 labeled quantum circuits from MQTBench, Qiskit Library, and synthetic random circuits.  
**Target**: Predict `improvement_ratio` = (original_depth - optimized_depth) / original_depth  
**Hardware**: IBM Brisbane (FakeBrisbane backend)  
**Runtime**: Kaggle T4 GPU (free tier)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports and Setup                                     ║
# ╚══════════════════════════════════════════════════════════════════╝

!pip install -q torch-geometric torch-scatter torch-sparse qiskit

import torch
import json
import time
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
from torch.nn import Linear, ReLU, Dropout, MSELoss, ELU
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# ── GPU Setup ──────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    print(f"🚀 Using {torch.cuda.device_count()} GPUs!")
    multi_gpu = True
else:
    print(f"Using: {device}")
    multi_gpu = False

# Print GPU info
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} — {props.total_mem / 1024**3:.1f} GB")
else:
    print("  ⚠️  No GPU detected, training will be slow")

print(f"\nPyTorch: {torch.__version__}")
print("✅ All imports successful")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Gate Feature Encoder                                  ║
# ╚══════════════════════════════════════════════════════════════════╝

# 20 gate types for one-hot encoding
GATE_TYPES = [
    'h', 'cx', 'rz', 'ry', 'rx', 'x', 'y', 'z', 'swap', 'ccx',
    't', 'tdg', 's', 'sdg', 'measure', 'barrier', 'reset', 'u', 'p', 'unknown'
]
GATE_TO_IDX = {g: i for i, g in enumerate(GATE_TYPES)}
NUM_GATE_TYPES = len(GATE_TYPES)
FEATURE_DIM = NUM_GATE_TYPES + 1  # 20 one-hot + 1 normalized qubit index = 21


def gate_to_feature(gate_name: str, qubit_indices: list, circuit_num_qubits: int) -> torch.Tensor:
    """Encode a gate as a 21-dimensional feature vector.
    
    Features:
        [0:20]  — One-hot encoding of gate type
        [20]    — Normalized qubit index (first qubit / total qubits)
    """
    # One-hot encode gate type
    one_hot = torch.zeros(NUM_GATE_TYPES)
    gate_lower = gate_name.lower()
    idx = GATE_TO_IDX.get(gate_lower, GATE_TO_IDX['unknown'])
    one_hot[idx] = 1.0
    
    # Normalized qubit position (0 to 1)
    if qubit_indices and circuit_num_qubits > 0:
        norm_qubit = qubit_indices[0] / circuit_num_qubits
    else:
        norm_qubit = 0.0
    
    feature = torch.cat([one_hot, torch.tensor([norm_qubit])])
    return feature


print(f"Feature dimension: {FEATURE_DIM}")
print(f"Gate types: {NUM_GATE_TYPES}")
print(f"Example: gate_to_feature('cx', [0, 1], 5) = {gate_to_feature('cx', [0, 1], 5)}")
print("✅ Gate encoder ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Circuit to Graph Converter                            ║
# ╚══════════════════════════════════════════════════════════════════╝

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_dag


def circuit_qasm_to_graph(qasm_str: str, improvement_ratio: float) -> Data:
    """Convert a QASM circuit string to a PyG Data graph.
    
    Steps:
        1. Parse QASM → QuantumCircuit
        2. Convert to DAG (directed acyclic graph)
        3. Each gate → node with 21-dim feature vector
        4. Each dependency → directed edge
        5. Label y = improvement_ratio
    
    Returns:
        PyG Data object, or None if parsing fails.
    """
    # Parse QASM string
    try:
        if qasm_str.strip().startswith('OPENQASM 3'):
            from qiskit.qasm3 import loads as qasm_loads
            circuit = qasm_loads(qasm_str)
        else:
            try:
                circuit = QuantumCircuit.from_qasm_str(qasm_str)
            except Exception:
                from qiskit.qasm2 import loads as qasm2_loads
                circuit = qasm2_loads(qasm_str)
    except Exception:
        return None
    
    # Convert to DAG
    try:
        dag = circuit_to_dag(circuit)
    except Exception:
        return None
    
    num_qubits = circuit.num_qubits
    
    # Collect operation nodes (skip input/output nodes)
    op_nodes = list(dag.op_nodes())
    
    if len(op_nodes) == 0:
        return None
    
    # Map each op node to an index
    node_to_idx = {node._node_id: i for i, node in enumerate(op_nodes)}
    
    # Build node features
    features = []
    for node in op_nodes:
        qubit_indices = [circuit.qubits.index(q) for q in node.qargs]
        feat = gate_to_feature(node.op.name, qubit_indices, num_qubits)
        features.append(feat)
    
    x = torch.stack(features)
    
    # Build edges from DAG dependencies
    edge_src = []
    edge_dst = []
    for edge in dag.edges():
        src_id = edge[0]._node_id if hasattr(edge[0], '_node_id') else None
        dst_id = edge[1]._node_id if hasattr(edge[1], '_node_id') else None
        if src_id in node_to_idx and dst_id in node_to_idx:
            edge_src.append(node_to_idx[src_id])
            edge_dst.append(node_to_idx[dst_id])
    
    # Handle circuits with no edges — add self-loops
    if len(edge_src) == 0:
        edge_src = list(range(len(op_nodes)))
        edge_dst = list(range(len(op_nodes)))
    
    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    y = torch.tensor([improvement_ratio], dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, y=y)


# Quick test
test_qasm = '''OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
h q[0];
cx q[0],q[1];
cx q[1],q[2];
'''
test_graph = circuit_qasm_to_graph(test_qasm, 0.5)
if test_graph is not None:
    print(f"Test graph: {test_graph.num_nodes} nodes, {test_graph.num_edges} edges")
    print(f"Features shape: {test_graph.x.shape}")
    print(f"Label: {test_graph.y}")
    print("✅ Circuit-to-graph converter working")
else:
    print("❌ Test conversion failed")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Build Full Dataset                                    ║
# ╚══════════════════════════════════════════════════════════════════╝

DATASET_PATH = '/kaggle/input/quantum-circuit-dataset/dataset_clean.json'
BATCH_SIZE = 128  # Larger batch for dual T4 GPUs

print(f"📂 Loading dataset from {DATASET_PATH}...")
with open(DATASET_PATH) as f:
    raw_dataset = json.load(f)
print(f"   Loaded {len(raw_dataset)} circuit records\n")

# Convert all circuits to PyG graphs
graphs = []
failed = 0
start_time = time.time()

for i, record in enumerate(raw_dataset):
    graph = circuit_qasm_to_graph(
        record['original_qasm'],
        record['improvement_ratio']
    )
    if graph is not None:
        graphs.append(graph)
    else:
        failed += 1
    
    if (i + 1) % 500 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        print(f"  ✓ {i+1}/{len(raw_dataset)} processed — "
              f"{len(graphs)} graphs, {failed} failed [{rate:.0f}/s]")

elapsed = time.time() - start_time
print(f"\n📊 Conversion complete in {elapsed:.1f}s")
print(f"   ✅ {len(graphs)} valid graphs")
print(f"   ❌ {failed} failed conversions")

# ── Train / Val / Test Split ───────────────────────────────────────
train_graphs, temp_graphs = train_test_split(graphs, test_size=0.30, random_state=42)
val_graphs, test_graphs = train_test_split(temp_graphs, test_size=0.50, random_state=42)

print(f"\n📋 Dataset split:")
print(f"   Train: {len(train_graphs)} graphs (70%)")
print(f"   Val:   {len(val_graphs)} graphs (15%)")
print(f"   Test:  {len(test_graphs)} graphs (15%)")

# ── DataLoaders ────────────────────────────────────────────────────
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")
print("\n✅ Dataset ready for training")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — GNN Model Architecture                                ║
# ╚══════════════════════════════════════════════════════════════════╝

class QuantumGNN(torch.nn.Module):
    """Graph Attention Network for predicting quantum circuit optimization ratios.
    
    Architecture:
        3 GAT layers → global_mean_pool → 3 FC layers → sigmoid → improvement_ratio
    """
    
    def __init__(self, input_dim=21, hidden_dim=64, output_dim=1):
        super(QuantumGNN, self).__init__()
        
        # GAT layers with multi-head attention
        self.conv1 = GATConv(input_dim, hidden_dim, heads=4, concat=True)   # → 256
        self.conv2 = GATConv(hidden_dim * 4, hidden_dim, heads=4, concat=True)  # → 256
        self.conv3 = GATConv(hidden_dim * 4, 32, heads=1, concat=True)      # → 32
        
        # Activation and regularization
        self.elu = ELU()
        self.dropout = Dropout(0.3)
        
        # Fully connected head
        self.fc1 = Linear(32, 64)
        self.fc2 = Linear(64, 32)
        self.fc3 = Linear(32, output_dim)
        
        self.relu = ReLU()
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        # GAT message passing
        x = self.elu(self.conv1(x, edge_index))
        x = self.elu(self.conv2(x, edge_index))
        x = self.elu(self.conv3(x, edge_index))
        
        # Graph-level readout
        x = global_mean_pool(x, batch)
        
        # FC head
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        
        # Sigmoid — improvement ratio is between 0 and 1
        x = torch.sigmoid(x)
        return x.squeeze(-1)


# ── Instantiate and print ──────────────────────────────────────────
model = QuantumGNN(input_dim=FEATURE_DIM)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: QuantumGNN (GAT)")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Input dim:  {FEATURE_DIM}")
print(f"  Hidden dim: 64")
print(f"  GAT heads:  4 → 4 → 1")
print(f"  Output:     sigmoid → [0, 1]")
print()
print(model)
print("\n✅ Model architecture defined")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Training Loop                                         ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Multi-GPU wrapping ─────────────────────────────────────────────
if multi_gpu:
    model = torch.nn.DataParallel(model)
model = model.to(device)

# ── Training config ────────────────────────────────────────────────
NUM_EPOCHS = 100
LEARNING_RATE = 0.001
PATIENCE = 15  # Early stopping patience
SAVE_PATH = '/kaggle/working/gnn_best.pt'

optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5, verbose=True)
criterion = MSELoss()

# ── Training state ─────────────────────────────────────────────────
train_losses = []
val_losses = []
val_maes = []
best_val_loss = float('inf')
patience_counter = 0
best_epoch = 0

print(f"Training config:")
print(f"  Epochs:     {NUM_EPOCHS}")
print(f"  LR:         {LEARNING_RATE}")
print(f"  Patience:   {PATIENCE}")
print(f"  Criterion:  MSELoss")
print(f"  Scheduler:  ReduceLROnPlateau (patience=5)")
print(f"  Device:     {device}")
print(f"  Save path:  {SAVE_PATH}")
print(f"\n{'='*70}")
print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>12} {'Val MAE':>10} {'LR':>10} {'Status':>8}")
print(f"{'-'*70}")

train_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────
    model.train()
    total_train_loss = 0
    num_train_batches = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)
        loss = criterion(pred, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()
        num_train_batches += 1
    
    avg_train_loss = total_train_loss / num_train_batches
    train_losses.append(avg_train_loss)
    
    # ── Validate ───────────────────────────────────────────────────
    model.eval()
    total_val_loss = 0
    total_val_mae = 0
    num_val_samples = 0
    num_val_batches = 0
    
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            pred = model(batch)
            loss = criterion(pred, batch.y)
            total_val_loss += loss.item()
            total_val_mae += torch.abs(pred - batch.y).sum().item()
            num_val_samples += batch.y.size(0)
            num_val_batches += 1
    
    avg_val_loss = total_val_loss / num_val_batches
    avg_val_mae = total_val_mae / num_val_samples
    val_losses.append(avg_val_loss)
    val_maes.append(avg_val_mae)
    
    # ── LR Scheduler ───────────────────────────────────────────────
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(avg_val_loss)
    
    # ── Best model & early stopping ────────────────────────────────
    status = ''
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch
        patience_counter = 0
        # Save best weights
        state_dict = model.module.state_dict() if multi_gpu else model.state_dict()
        torch.save(state_dict, SAVE_PATH)
        status = '⭐ best'
    else:
        patience_counter += 1
    
    # ── Print progress ─────────────────────────────────────────────
    print(f"{epoch:>6} {avg_train_loss:>12.6f} {avg_val_loss:>12.6f} "
          f"{avg_val_mae:>10.4f} {current_lr:>10.6f} {status:>8}")
    
    # ── Early stopping ─────────────────────────────────────────────
    if patience_counter >= PATIENCE:
        print(f"\n⏹  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

train_time = time.time() - train_start
print(f"{'='*70}")
print(f"\n✅ Training complete in {train_time/60:.1f} minutes")
print(f"   Best epoch: {best_epoch} — Val Loss: {best_val_loss:.6f}")
print(f"   Best Val MAE: {val_maes[best_epoch-1]:.4f} ({val_maes[best_epoch-1]*100:.1f}%)")
print(f"   Model saved to {SAVE_PATH}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Training Curves                                       ║
# ╚══════════════════════════════════════════════════════════════════╝

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(train_losses) + 1)

# ── Left: Train vs Val Loss ────────────────────────────────────────
ax1.plot(epochs_range, train_losses, label='Train Loss', color='#2563eb', linewidth=2)
ax1.plot(epochs_range, val_losses, label='Val Loss', color='#dc2626', linewidth=2)
ax1.axvline(x=best_epoch, color='#16a34a', linestyle='--', alpha=0.7, label=f'Best (epoch {best_epoch})')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('MSE Loss', fontsize=12)
ax1.set_title('Training vs Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# ── Right: Val MAE ─────────────────────────────────────────────────
ax2.plot(epochs_range, [m * 100 for m in val_maes], color='#7c3aed', linewidth=2)
ax2.axvline(x=best_epoch, color='#16a34a', linestyle='--', alpha=0.7, label=f'Best (epoch {best_epoch})')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('MAE (%)', fontsize=12)
ax2.set_title('Validation MAE (percentage points)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Training curves saved to /kaggle/working/training_curves.png")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Test Evaluation                                       ║
# ╚══════════════════════════════════════════════════════════════════╝

# Load best model weights
best_model = QuantumGNN(input_dim=FEATURE_DIM)
best_model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
best_model = best_model.to(device)
best_model.eval()

print("📊 Running test evaluation...\n")

all_preds = []
all_actuals = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred = best_model(batch)
        all_preds.extend(pred.cpu().numpy().tolist())
        all_actuals.extend(batch.y.cpu().numpy().tolist())

all_preds = np.array(all_preds)
all_actuals = np.array(all_actuals)

# ── Compute metrics ────────────────────────────────────────────────
test_mse = float(np.mean((all_preds - all_actuals) ** 2))
test_mae = float(np.mean(np.abs(all_preds - all_actuals)))
mean_pred = float(np.mean(all_preds))
mean_actual = float(np.mean(all_actuals))
closeness = float(np.mean(np.abs(all_preds - all_actuals))) * 100

print(f"TEST SET RESULTS ({len(all_preds)} circuits)")
print(f"{'='*45}")
print(f"  Test MSE:                      {test_mse:.6f}")
print(f"  Test MAE:                      {test_mae:.4f} ({test_mae*100:.1f}%)")
print(f"  Mean predicted improvement:    {mean_pred*100:.1f}%")
print(f"  Mean actual improvement:       {mean_actual*100:.1f}%")
print(f"  Avg prediction error:          {closeness:.1f} percentage points")
print(f"{'='*45}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Benchmark vs Qiskit                                   ║
# ╚══════════════════════════════════════════════════════════════════╝

print("🔬 Benchmarking GNN predictions against actual Qiskit results...\n")

# Use first 100 test samples (or all if < 100)
n_benchmark = min(100, len(all_preds))
bench_preds = all_preds[:n_benchmark]
bench_actuals = all_actuals[:n_benchmark]

# Calculate accuracies at different thresholds
errors = np.abs(bench_preds - bench_actuals)

accuracy_10 = float(np.mean(errors <= 0.10) * 100)  # within 10% abs
accuracy_20 = float(np.mean(errors <= 0.20) * 100)  # within 20% abs
mean_error_pct = float(np.mean(errors) * 100)        # in percentage points
avg_actual_pct = float(np.mean(bench_actuals) * 100)
avg_pred_pct = float(np.mean(bench_preds) * 100)

print(f"GNN BENCHMARK RESULTS")
print(f"=====================")
print(f"Prediction accuracy (within 10%): {accuracy_10:.0f}%")
print(f"Prediction accuracy (within 20%): {accuracy_20:.0f}%")
print(f"Mean prediction error: {mean_error_pct:.1f} percentage points")
print(f"Average actual improvement: {avg_actual_pct:.1f}%")
print(f"Average predicted improvement: {avg_pred_pct:.1f}%")
print()

# ── Scatter plot: predicted vs actual ──────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(bench_actuals * 100, bench_preds * 100, alpha=0.6, s=30, color='#7c3aed')
ax.plot([0, 100], [0, 100], '--', color='#dc2626', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Improvement (%)', fontsize=12)
ax.set_ylabel('Predicted Improvement (%)', fontsize=12)
ax.set_title('GNN Predictions vs Actual Qiskit Optimization', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-5, 105)
ax.set_ylim(-5, 105)
plt.tight_layout()
plt.savefig('/kaggle/working/benchmark_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Benchmark complete")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Save Everything                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

import datetime

OUTPUT_DIR = '/kaggle/working'

# ── 1. Benchmark results ───────────────────────────────────────────
benchmark_results = {
    'test_mse': test_mse,
    'test_mae': test_mae,
    'test_mae_pct': test_mae * 100,
    'accuracy_within_10pct': accuracy_10,
    'accuracy_within_20pct': accuracy_20,
    'mean_error_pct_points': mean_error_pct,
    'avg_actual_improvement_pct': avg_actual_pct,
    'avg_predicted_improvement_pct': avg_pred_pct,
    'n_test_samples': len(all_preds),
    'n_benchmark_samples': n_benchmark,
}
with open(f'{OUTPUT_DIR}/benchmark_results.json', 'w') as f:
    json.dump(benchmark_results, f, indent=2)

# ── 2. Model config ────────────────────────────────────────────────
model_config = {
    'input_dim': FEATURE_DIM,
    'hidden_dim': 64,
    'output_dim': 1,
    'num_layers': 3,
    'architecture': 'GAT',
    'gat_heads': [4, 4, 1],
    'training_circuits': len(train_graphs),
    'val_circuits': len(val_graphs),
    'test_circuits': len(test_graphs),
    'total_parameters': total_params,
    'best_epoch': best_epoch,
    'val_mae': round(val_maes[best_epoch - 1], 4),
    'test_mae': round(test_mae, 4),
    'avg_improvement_predicted': round(mean_pred * 100, 1),
    'trained_on': 'MQTBench + Qiskit Library + Synthetic',
    'target_hardware': 'IBM Brisbane',
    'training_time_minutes': round(train_time / 60, 1),
    'trained_at': datetime.datetime.now().isoformat(),
}
with open(f'{OUTPUT_DIR}/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

# ── 3. Verify all files saved ──────────────────────────────────────
import os
saved_files = [
    'gnn_best.pt',
    'training_curves.png',
    'benchmark_scatter.png',
    'benchmark_results.json',
    'model_config.json',
]

print("📁 Saved files:")
for fname in saved_files:
    fpath = f'{OUTPUT_DIR}/{fname}'
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  ✅ {fname} ({size_mb:.2f} MB)")
    else:
        print(f"  ❌ {fname} — NOT FOUND")

# ── Final summary ──────────────────────────────────────────────────
best_val_mae_pct = val_maes[best_epoch - 1] * 100
test_mae_pct = test_mae * 100

print(f"\n{'='*45}")
print(f"  MODEL READY FOR quantumopt PACKAGE")
print(f"{'='*45}")
print(f"  Weights saved:  gnn_best.pt")
print(f"  Val MAE:        {best_val_mae_pct:.2f}%")
print(f"  Test MAE:       {test_mae_pct:.2f}%")
print(f"  Avg improvement predicted: {mean_pred*100:.1f}%")
print(f"  Copy gnn_best.pt to quantumopt/models/weights/")
print(f"{'='*45}")